# Sheet 3.4: Fine-tuning: LoRA, QLoRA, data and evaluation

**Roadmap topic 3.4** (Fanout 10.01-10.05). Read first: the LoRA and QLoRA papers, Thinking Machines' *LoRA Without Regret*, *LoRA Learns Less and Forgets Less*, and the TRL `SFTTrainer` and PEFT docs.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Exercises 1-4 run on CPU; Exercise 5 is a Colab T4 project.

In [ ]:
import math, re, random, hashlib, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

## Exercise 1: LoRA from scratch

`LoRALinear(base, r, alpha, dropout=0.0)` wraps a frozen `nn.Linear`: `y = base(x) + (alpha/r)·B(A(dropout(x)))` with `A` initialised Kaiming-uniform (`a=√5`, like PEFT) and `B` zeros, so the wrapped layer starts identical to the base. Add `merge()` (fold the update into `base.weight`, then the adapter path is skipped) and `unmerge()`.

`apply_lora(model, target_names, r, alpha)` replaces every `nn.Linear` whose attribute name is in `target_names` (e.g. `{"q_proj", "v_proj"}`) and freezes everything except the LoRA matrices. `count_trainable(model)` returns `(trainable, total)`.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        # YOUR CODE
        raise NotImplementedError
    def forward(self, x): raise NotImplementedError
    def merge(self): raise NotImplementedError
    def unmerge(self): raise NotImplementedError
def apply_lora(model, target_names, r=8, alpha=16):
    # YOUR CODE
    raise NotImplementedError
def count_trainable(model):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    torch.manual_seed(0)
    class Attn(nn.Module):
        def __init__(s): super().__init__(); s.q_proj, s.k_proj, s.v_proj, s.o_proj = (nn.Linear(64, 64) for _ in range(4))
        def forward(s, x): return s.o_proj(s.q_proj(x) * torch.sigmoid(s.k_proj(x)) + s.v_proj(x))
    m = nn.Sequential(Attn(), nn.GELU(), Attn()); x = torch.randn(10, 64); y0 = m(x).detach()
    apply_lora(m, {"q_proj", "v_proj"}, r=4, alpha=8)
    assert torch.allclose(m(x), y0), "a fresh LoRA model must equal the base model"
    tr, tot = count_trainable(m)
    assert tr == 4 * (4 * 64 + 64 * 4), f"trainable params {tr}"
    opt = torch.optim.SGD([p for p in m.parameters() if p.requires_grad], lr=0.5)
    for _ in range(3): opt.zero_grad(); m(x).pow(2).mean().backward(); opt.step()
    y1 = m(x).detach(); assert not torch.allclose(y1, y0), "training should change the output"
    for mod in m.modules():
        if isinstance(mod, LoRALinear): mod.merge()
    assert torch.allclose(m(x), y1, atol=1e-5), "merged output differs"
    for mod in m.modules():
        if isinstance(mod, LoRALinear): mod.unmerge()
    assert torch.allclose(m(x), y1, atol=1e-5), "unmerge must restore the adapter path"
    assert all(not p.requires_grad for n, p in m.named_parameters() if "base" in n)
    print(f"PASS  trainable {tr:,} of {tot:,} parameters ({tr / tot:.2%})")
_check1()

**Written (1a):** Why is `B` initialised to zero and not `A`? What would go wrong if both were random? And why does the `alpha/r` scaling make it easier to change `r` without retuning the learning rate?

**Your answer:**

## Exercise 2: Chat formatting with assistant-only loss

You'll use a toy character-level tokenizer so the check can run offline. The chat template is:
```
<|user|>\n{content}<|end|>\n<|assistant|>\n{content}<|end|>\n   (repeated per turn)
```
`build_example(messages, tokenize, max_len)` returns `(input_ids, labels)` where labels are `-100` everywhere **except** the tokens of each assistant reply **and its closing `<|end|>`** (the model must learn to stop). If longer than `max_len`, drop the **earliest whole messages** until it fits (never cut a message in half); return `None` if even the final reply doesn't fit.

In [ ]:
SPECIAL = ["<|user|>", "<|assistant|>", "<|end|>"]
def tokenize(text):
    out, i = [], 0
    while i < len(text):
        for j, s in enumerate(SPECIAL):
            if text.startswith(s, i): out.append(1000 + j); i += len(s); break
        else: out.append(ord(text[i])); i += 1
    return out

In [ ]:
def build_example(messages, tokenize, max_len=10_000):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    msgs = [{"role": "user", "content": "2+2?"}, {"role": "assistant", "content": "4"},
            {"role": "user", "content": "and 3+3?"}, {"role": "assistant", "content": "6"}]
    ids, labels = build_example(msgs, tokenize)
    full = "<|user|>\n2+2?<|end|>\n<|assistant|>\n4<|end|>\n<|user|>\nand 3+3?<|end|>\n<|assistant|>\n6<|end|>\n"
    assert ids == tokenize(full), "template wrong"
    trained = [t for t in labels if t != -100]
    assert trained == [ord("4"), 1002, ord("6"), 1002], f"trained tokens {trained}"
    assert len(ids) == len(labels)
    short = build_example(msgs, tokenize, max_len=20)
    assert short is not None and short[0] == tokenize("<|user|>\nand 3+3?<|end|>\n<|assistant|>\n6<|end|>\n")[-len(short[0]):] and len(short[0]) <= 20
    assert build_example([{"role": "user", "content": "hi"}, {"role": "assistant", "content": "x" * 50}], tokenize, max_len=20) is None
    print("PASS")
_check2()

**Written (2a):** A colleague's SFT run trains on prompt tokens too. Their loss is lower than yours and their model is worse. Explain both observations.

**Your answer:**

## Exercise 3: Near-duplicate removal with MinHash

Fine-tuning sets scraped from the web are full of near-duplicates. Implement:
- `shingles(text, k=5)`: set of word 5-grams (lowercased, punctuation stripped).
- `minhash(sh, num_perm=128, seed=0)`: signature using `num_perm` hash functions of the form `(a·h(x) + b) mod p`, with `h` a stable 64-bit hash (e.g. from `hashlib.blake2b`) and `p = 2^61 - 1`.
- `est_jaccard(sig1, sig2)`.
- `dedup(texts, threshold=0.8)`: indices to keep (first occurrence wins). O(n²) comparison of signatures is fine for this size.

In [ ]:
def shingles(text, k=5): raise NotImplementedError
def minhash(sh, num_perm=128, seed=0): raise NotImplementedError
def est_jaccard(s1, s2): raise NotImplementedError
def dedup(texts, threshold=0.8): raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    r = random.Random(1); vocab = [f"w{i}" for i in range(500)]
    base = [" ".join(r.choice(vocab) for _ in range(120)) for _ in range(20)]
    def perturb(t, n):
        w = t.split()
        for _ in range(n): w[r.randrange(len(w))] = r.choice(vocab)
        return " ".join(w)
    a, b = base[0], perturb(base[0], 6)
    true = len(shingles(a) & shingles(b)) / len(shingles(a) | shingles(b))
    est = est_jaccard(minhash(shingles(a)), minhash(shingles(b)))
    assert abs(est - true) < 0.1, f"estimate {est:.2f} vs true {true:.2f}"
    texts = base + [perturb(base[i], 1) for i in range(10)] + [base[3].upper() + "!!"]
    keep = dedup(texts)
    assert keep == list(range(20)), f"kept {keep}; near-duplicates (indices 20-30) should be removed"
    print(f"PASS  Jaccard true {true:.2f}, estimated {est:.2f}")
_check3()

## Exercise 4: Benchmark contamination check

`contaminated(train_texts, eval_texts, n=13)` returns the indices of eval examples sharing at least one word `n`-gram with any training text (the GPT-3 paper's method). Use the same normalisation as `shingles`. Build an index of training n-grams once (a set), then scan eval items.

In [ ]:
def contaminated(train_texts, eval_texts, n=13):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    r = random.Random(2); vocab = [f"t{i}" for i in range(2000)]
    ev = [" ".join(r.choice(vocab) for _ in range(40)) for _ in range(30)]
    tr = [" ".join(r.choice(vocab) for _ in range(300)) for _ in range(50)]
    leak = ev[7].split()[5:20]; tr[10] = tr[10] + " " + " ".join(leak).upper() + ". more text"
    tr[11] = tr[11] + " " + " ".join(ev[12].split()[:12])           # only 12 words: below threshold
    assert contaminated(tr, ev) == [7], contaminated(tr, ev)
    print("PASS")
_check4()

---
## Exercise 5 (Colab T4): QLoRA SFT, evaluation and deployment

1. **Task.** Pick a narrow, checkable task: e.g. text-to-SQL on `b-mc2/sql-create-context`, or JSON extraction. Hold out 500 test examples. Run your Exercise 3 dedup and Exercise 4 contamination check on train vs test and report the numbers.
2. **Baseline.** Evaluate `Qwen/Qwen2.5-1.5B-Instruct` zero-shot and 3-shot with exact-match (or execution accuracy for SQL).
3. **QLoRA SFT.** 4-bit NF4 base (`bitsandbytes`), LoRA on all linear layers, TRL `SFTTrainer` with assistant-only loss. Sweep `r ∈ {4, 16, 64}` × 3 learning rates (short runs). Use *LoRA Without Regret*'s LR guidance as your starting point.
4. **Forgetting.** For every run, also report a general benchmark subset with `lm-evaluation-harness` (e.g. 500 questions of `mmlu` or `arc_easy`). Plot target-task gain vs general-task change.
5. **Deploy.** Merge the best adapter, convert to GGUF with llama.cpp, quantise to Q4_K_M, run it on your laptop. Report tokens/sec, time-to-first-token and exact-match vs the fp16 merged model on 100 test prompts.

**Targets:** the best fine-tune beats 3-shot by ≥ 10 points on your task; quantisation costs ≤ 2 points.

## Checkpoint
A stakeholder wants to fine-tune a model on 300 support tickets so it "knows our product". Write the half-page plan you'd actually propose, including whether to fine-tune at all (vs RAG or prompting), how you'd evaluate, and what could go wrong.

---
### Hints for self-grading
- 1a: with B = 0 the update BA starts at zero, so training starts from the pretrained model; gradients still reach B through A's random projection. Both random would add a random perturbation to the pretrained weights at step 0. `alpha/r` keeps the update's scale roughly constant as `r` changes.
- 2a: prompt tokens are easy to predict (often copied or templated), which lowers the average loss; but the gradient is partly spent on modelling user text instead of answers, and the model may learn to continue as the user.